The purpose of this script is to add a boolean column to session_times.csv in order to indicate whether the given session is a special session.

In [5]:
import pandas as pd

df_raw = pd.read_excel(r'C:\Users\Tillman\Thesis\Datasets\IISE 2024\IISE2024_Raw.xlsx')
df_raw.head(5)

,Unnamed: 0,favoriteItem,SlotTime,SlotDate,ParentList,field_Abstract,RoleListItem,Paper Number
0,0,A Study on Changes in the Interior Environment...,NaN,NaN,Program: Oral Presentations\nTrack: Human Fact...,Abstract\nThis study aims to predict future ch...,NaN,5100
1,0,Optimizing Operations Managerial Performance: ...,12:00 PM - 12:20 PM,"Sunday, May 19, 2024",Session: Optimization\nProgram: Oral Presentat...,Abstract\nSenior company leadership at an airc...,Donald Fleming\nThe Boeing CompanyKareem Badr\...,5113
2,0,Discovering Puerto Rico Roads Profile Influenc...,2:10 PM - 2:30 PM,"Monday, May 20, 2024",Session: Logistics I\nProgram: Oral Presentati...,Abstract\nThe infrastructure of Puerto Rico (P...,Yesenia Cruz Cantillo\nUniversidad Ana G. Ménd...,5130
3,0,Designing Manufacturing Enterprises System for...,8:00 AM - 8:20 AM,"Monday, May 20, 2024",Session: Process Planning-I\nProgram: Oral Pre...,"Abstract\nCurrently, researchers are concentra...",Ibrahim Garbie\nHelwan UniversityMuhammad Garb...,5139
4,0,Achieving Equitable Access to Medical Laborato...,12:20 PM - 12:40 PM,"Monday, May 20, 2024",Session: Social Good Analytics\nProgram: Oral ...,"Abstract\nIn contemporary healthcare, laborato...",Yiqun Jiang\nIowa State UniversityWenli Zhang\...,5151


In [10]:
df_session_times = pd.read_csv(r'C:\Users\Tillman\Thesis\Datasets\IISE 2024\session_times.csv')
df_session_times

,session_name,start_datetime,end_datetime,session_id
0,AI in Health Systems,5/19/2024 12:00,5/19/2024 13:20,1
1,Advanced Data Analytics for Quality Control an...,5/19/2024 15:00,5/19/2024 16:20,2
2,Advanced Simulation Models,5/19/2024 16:30,5/19/2024 17:50,3
3,Advanced Topics in Smart Manufacturing I,5/19/2024 13:30,5/19/2024 14:50,4
4,Advanced Topics of QCRE Applications I,5/21/2024 8:00,5/21/2024 9:20,5
...,...,...,...,...
237,Warehousing,5/19/2024 16:30,5/19/2024 17:50,238
238,What Leaders Look For In Promoting IE’s Into M...,5/20/2024 13:30,5/20/2024 14:50,239
239,Work Systems I,5/19/2024 12:00,5/19/2024 13:00,240
240,Work environment in health systems,5/19/2024 15:00,5/19/2024 16:20,241


In [20]:
def fuzzy_left_join(left_df, right_df, left_col, right_col):
    """
    Performs a fuzzy left join where rows are matched if the text in the 
    specified column of the left DataFrame is contained within the text 
    in the specified column of the right DataFrame.

    Args:
        left_df: The left DataFrame.
        right_df: The right DataFrame.
        left_col: The name of the column in the left DataFrame to join on.
        right_col: The name of the column in the right DataFrame to join on.

    Returns:
        A new DataFrame with the joined data, or the original left DataFrame if no matches are found.
        Returns an empty DataFrame if either input DataFrame is empty.
    """

    if left_df.empty or right_df.empty:
        return pd.DataFrame() # Return empty DataFrame if either input is empty

    merged_rows = []
    for _, left_row in left_df.iterrows():
        left_val = left_row[left_col]
        match_found = False

        for _, right_row in right_df.iterrows():
            right_val = right_row[right_col]
            if isinstance(left_val, str) and isinstance(right_val, str) and left_val in right_val:
                merged_row = left_row.to_dict()  # Start with left row data
                merged_row.update(right_row.to_dict()) # Add right row data, overwriting if keys are the same
                merged_rows.append(merged_row)
                match_found = True
                break  # Stop searching for matches once one is found

        if not match_found: # If no match is found, append the left row data to the result
            merged_rows.append(left_row.to_dict())

    return pd.DataFrame(merged_rows)

merged_df = fuzzy_left_join(df_session_times, df_raw, 'session_name', 'ParentList')
merged_df.head(5)

,session_name,start_datetime,end_datetime,session_id,Unnamed: 0,favoriteItem,SlotTime,SlotDate,ParentList,field_Abstract,RoleListItem,Paper Number
0,AI in Health Systems,5/19/2024 12:00,5/19/2024 13:20,1,0,Advancing Acute Kidney Injury Management throu...,12:00 PM - 12:20 PM,"Sunday, May 19, 2024",Session: AI in Health Systems\nProgram: Oral P...,Abstract\nAcute Kidney Injury (AKI) is a preva...,Dima Al Absi\nKhalifa University of Science an...,6437
1,Advanced Data Analytics for Quality Control an...,5/19/2024 15:00,5/19/2024 16:20,2,0,Simultaneous Long-tailed Learning and Multi-mo...,3:00 PM - 3:20 PM,"Sunday, May 19, 2024",Session: Advanced Data Analytics for Quality C...,Abstract\nIt is common in quality control prob...,Y\nHeegeon YoonHeeyoung Kim,6277
2,Advanced Simulation Models,5/19/2024 16:30,5/19/2024 17:50,3,0,Teaching Simulation Using the Kotlin Simulatio...,4:30 PM - 4:50 PM,"Sunday, May 19, 2024",Session: Advanced Simulation Models\nProgram: ...,Abstract\nUsing the Kotlin programming languag...,Manuel Rossetti\nUnversity of Arkansas,5782
3,Advanced Topics in Smart Manufacturing I,5/19/2024 13:30,5/19/2024 14:50,4,0,GIFR: A Graph-Informed Functional Regression M...,1:30 PM - 1:50 PM,"Sunday, May 19, 2024",Session: Advanced Topics in Smart Manufacturin...,Abstract\nHigh Temperature Superconductors (HT...,"A\nKiran Adhikari\nIndustrial Engineering, Uni...",6596
4,Advanced Topics of QCRE Applications I,5/21/2024 8:00,5/21/2024 9:20,5,0,Strata Design in Stochastic Simulations with M...,8:20 AM - 8:40 AM,"Tuesday, May 21, 2024",Session: Advanced Topics of QCRE Applications ...,Abstract\nStratified sampling is one of the po...,Jaeshin Park\nDepartment of Industrial and Ope...,5922


In [21]:
merged_df = merged_df[['session_name','start_datetime','end_datetime','session_id','ParentList']]
merged_df.sort_values(by='ParentList', ascending=True, inplace=True)
merged_df['is_special_session'] = merged_df['ParentList'].apply(lambda x: 'Special Session' in x)
merged_df['is_special_and_panel_session'] = merged_df['is_special_session'] & merged_df['ParentList'].apply(lambda x: 'Panel' in x)
merged_df.drop(columns=['ParentList'], inplace=True)
merged_df.to_csv(r'C:\Users\Tillman\Thesis\Datasets\IISE 2024\session_times.csv')